# Day 10 Tutorial — Data Modeling for Analytics

**Goal:** Star schema, facts/dims, SCD interview fluency.


## Star schema
- Fact = events + measures + FKs
- Dimensions = descriptive context
- SCD1 overwrite; SCD2 history with validity columns


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
spark.createDataFrame([(1, 'Alice', 'East'), (2, 'Bob', 'West')],
    ['customer_key', 'customer_name', 'region']).createOrReplaceTempView('dim_customer')
spark.createDataFrame([(10, 'Tea', 'Beverages'), (20, 'Mug', 'Home')],
    ['product_key', 'product_name', 'category']).createOrReplaceTempView('dim_product')
spark.createDataFrame(
    [(100, 1, 10, 20240101, 2, 40.0), (101, 2, 20, 20240101, 1, 15.0),
     (102, 1, 20, 20240102, 3, 45.0)],
    ['sales_key', 'customer_key', 'product_key', 'date_key', 'qty', 'amount']
).createOrReplaceTempView('fact_sales')


In [ ]:
spark.sql('''
SELECT c.region, p.category, SUM(f.amount) AS revenue
FROM fact_sales f
JOIN dim_customer c ON f.customer_key = c.customer_key
JOIN dim_product p ON f.product_key = p.product_key
GROUP BY c.region, p.category
ORDER BY revenue DESC
''').show()
